# Olympiad Extension: A Mood Machine That Sees Shapes

Click a cell and press **Shift + Enter** to run it. Work from top to bottom. Cells marked **YOUR TURN** are for you to finish.

Run the first cell straight away: it draws 15,000 faces.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")
print("Using", device)

class EmojiFaces(Dataset):
    def __init__(self, how_many, first_seed, transform=None):
        self.faces = []
        for seed in range(first_seed, first_seed + how_many):
            picture, mood = make_face(seed)
            if transform:
                picture = transform(picture)
            self.faces.append((picture, mood))

    def __len__(self):
        return len(self.faces)

    def __getitem__(self, idx):
        return self.faces[idx]

get_ready = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("Drawing faces... (about a minute)")
train_faces = EmojiFaces(12000, first_seed=0, transform=get_ready)
test_faces = EmojiFaces(3000, first_seed=1_000_000, transform=get_ready)
train_loader = DataLoader(train_faces, batch_size=64, shuffle=True)
test_loader = DataLoader(test_faces, batch_size=64)
print(len(train_faces), "training faces and", len(test_faces), "test faces")

class MoodNet(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(3 * 32 * 32, hidden)
        self.hidden2 = nn.Linear(hidden, 128)
        self.output = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

def train_one_epoch(model, loader, loss_fn, optimiser):
    model.train()
    for faces, moods in loader:
        faces, moods = faces.to(device), moods.to(device)
        guesses = model(faces)            # 1. make guesses
        loss = loss_fn(guesses, moods)    # 2. how wrong were they?
        optimiser.zero_grad()             # 3. wipe old gradients
        loss.backward()                   # 4. find new gradients (back propagation)
        optimiser.step()                  # 5. nudge the weights
    return loss.item()

def check_accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for faces, moods in loader:
            faces, moods = faces.to(device), moods.to(device)
            guesses = model(faces).argmax(dim=1)
            correct += (guesses == moods).sum().item()
    return 100 * correct / len(loader.dataset)

from torch.utils.data import Subset
train_part = Subset(train_faces, range(10000))
val_part = Subset(train_faces, range(10000, 12000))
part_loader = DataLoader(train_part, batch_size=64, shuffle=True)
val_loader = DataLoader(val_part, batch_size=256)

def upside_down_accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for faces, moods in loader:
            faces = torch.flip(faces, dims=[2, 3]).to(device)    # turn every face upside down
            correct += (model(faces).argmax(dim=1) == moods.to(device)).sum().item()
    return 100 * correct / len(loader.dataset)

def count_numbers(model):
    return sum(p.numel() for p in model.parameters())

## 1. The Problem With Flatten
MoodNet unrolls each face into one long list, so it never knows which pixels are next to each other. Train it for 3 epochs, then test it upright and upside down.

In [ ]:
torch.manual_seed(1)
mlp = MoodNet().to(device)
opt = torch.optim.Adam(mlp.parameters(), lr=0.001)
for epoch in range(3):
    train_one_epoch(mlp, part_loader, nn.CrossEntropyLoss(), opt)
print("MoodNet:", count_numbers(mlp), "numbers to learn")
print("  validation, upright:     ", round(check_accuracy(mlp, val_loader), 1), "%")
print("  validation, upside down: ", round(upside_down_accuracy(mlp, val_loader), 1), "%")

## 2. One Filter
A **convolutional layer** slides a small grid of weights, called a **filter**, across the picture. At every position it multiplies and adds, just like a neuron. The same 9 weights are used everywhere, so a filter that finds an edge finds it anywhere in the picture.

Here is a filter we have set by hand. It responds where the picture gets brighter from top to bottom.

In [ ]:
edge = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    edge.weight[:] = torch.tensor([[[[-1.0, -1.0, -1.0], [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]]]])
face = train_faces[0][0]
grey = face.mean(dim=0, keepdim=True).unsqueeze(0)     # shape (1, 1, 32, 32)
found = edge(grey)[0, 0].detach()
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1); plt.imshow((face.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)); plt.axis("off"); plt.title("face")
plt.subplot(1, 2, 2); plt.imshow(found, cmap="gray"); plt.axis("off"); plt.title("what the filter finds")
plt.show()

In [ ]:
# YOUR TURN: change the filter so it finds edges from LEFT to RIGHT instead. Run it and compare.
with torch.no_grad():
    edge.weight[:] = torch.tensor([[[[-1.0, -1.0, -1.0], [0.0, 0.0, 0.0], [1.0, 1.0, 1.0]]]])
plt.imshow(edge(grey)[0, 0].detach(), cmap="gray"); plt.axis("off"); plt.show()

## 3. MoodCNN
In a CNN, the filters are not set by hand: they are learned, like any other weights.

- **Conv2d(3, 16, 3)**: 16 filters, each 3 x 3, looking at red, green and blue
- **MaxPool2d(2)**: keeps the biggest value in each 2 x 2 square, halving the width and height
- a second Conv2d and MaxPool2d, then Flatten and two Linear layers, as in MoodNet

In [ ]:
class MoodCNN(nn.Module):
    def __init__(self, channels=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),               # 32x32 -> 16x16
            nn.Conv2d(channels, 2 * channels, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 16x16 -> 8x8
        )
        self.classify = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * channels * 8 * 8, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classify(self.features(x))

print("MoodCNN:", count_numbers(MoodCNN()), "numbers to learn")
print("MoodNet:", count_numbers(MoodNet()), "numbers to learn")

## 4. Train and Compare
Three epochs, the same as MoodNet. This takes a little longer.

In [ ]:
def train_cnn(epochs=3, channels=16, flip_half=False, seed=1):
    torch.manual_seed(seed)
    net = MoodCNN(channels).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        net.train()
        for faces, moods in part_loader:
            if flip_half:
                turn = torch.rand(len(faces)) < 0.5                  # pick about half the batch
                faces[turn] = torch.flip(faces[turn], dims=[2, 3])   # and turn them upside down
            faces, moods = faces.to(device), moods.to(device)
            loss = loss_fn(net(faces), moods)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return net

cnn = train_cnn()
print("MoodCNN validation, upright:     ", round(check_accuracy(cnn, val_loader), 1), "%")
print("MoodCNN validation, upside down: ", round(upside_down_accuracy(cnn, val_loader), 1), "%")

## 5. Augmentation
The CNN still fails on upside-down faces, because it has never seen one. **Augmentation** changes the training pictures as you go, so the model sees more variety. Here, about half of every batch is turned upside down.

In [ ]:
cnn_aug = train_cnn(flip_half=True)
print("MoodCNN + augmentation, upright:     ", round(check_accuracy(cnn_aug, val_loader), 1), "%")
print("MoodCNN + augmentation, upside down: ", round(upside_down_accuracy(cnn_aug, val_loader), 1), "%")

## 6. Beyond Accuracy
A **confusion matrix** counts every pair of right mood and guessed mood. From it you can work out, for each mood:

- **precision**: of the faces it called this mood, how many really were?
- **recall**: of the faces that really were this mood, how many did it find?
- **F1**: one score that balances the two: 2 x precision x recall / (precision + recall)

In [ ]:
def confusion(model, loader):
    table = torch.zeros(10, 10, dtype=torch.int64)
    model.eval()
    with torch.no_grad():
        for faces, moods in loader:
            guesses = model(faces.to(device)).argmax(dim=1).cpu()
            for right, guess in zip(moods, guesses):
                table[right, guess] += 1
    return table

table = confusion(cnn_aug, val_loader)
plt.figure(figsize=(6, 5))
plt.imshow(table, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=90); plt.yticks(range(10), CLASSES)
plt.xlabel("guessed mood"); plt.ylabel("right mood"); plt.colorbar()
plt.show()

for m in range(10):
    tp = table[m, m].item()
    precision = tp / max(table[:, m].sum().item(), 1)
    recall = tp / max(table[m, :].sum().item(), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    print(CLASSES[m].ljust(10), " precision", round(precision, 2), " recall", round(recall, 2), " F1", round(f1, 2))

In [ ]:
# YOUR TURN: improve MoodCNN. Change ONE thing, compare on the validation faces, and record it. Ideas:
#   train_cnn(epochs=5, flip_half=True)
#   train_cnn(channels=32, flip_half=True)
#   add nn.Dropout(0.3) before the last Linear layer in MoodCNN, then train again